In [1]:
"""
MANUAL TRANSFORMER ENCODER / SELF-ATTENTION DEMONSTRATION
Based on the mathematical example in the supplied PDF.

Important:
- No NumPy, PyTorch, TensorFlow, SciPy, or other libraries are used.
- Matrix multiplication, transpose, softmax, sqrt, exp, tanh, GELU,
  and LayerNorm are implemented manually.
- The numerical values are intentionally kept close to the PDF.
"""

# ------------------------------------------------------------
# 0. Helper functions -- all implemented manually
# ------------------------------------------------------------

def zeros(rows, cols):
    return [[0.0 for _ in range(cols)] for _ in range(rows)]


def matmul(A, B):
    """Manual matrix multiplication: C = A x B."""
    rows_A = len(A)
    cols_A = len(A[0])
    rows_B = len(B)
    cols_B = len(B[0])

    if cols_A != rows_B:
        raise ValueError("Incompatible matrix dimensions.")

    C = zeros(rows_A, cols_B)

    for i in range(rows_A):
        for j in range(cols_B):
            total = 0.0
            for k in range(cols_A):
                total += A[i][k] * B[k][j]
            C[i][j] = total

    return C


def transpose(A):
    """Manual transpose."""
    return [list(row) for row in zip(*A)]


def vector_add(a, b):
    return [a[i] + b[i] for i in range(len(a))]


def matrix_add(A, B):
    return [
        [A[i][j] + B[i][j] for j in range(len(A[0]))]
        for i in range(len(A))
    ]


def matrix_scale(A, scalar):
    return [[x * scalar for x in row] for row in A]


# ------------------------------------------------------------
# Basic numerical functions WITHOUT importing math
# ------------------------------------------------------------

def absolute(x):
    return -x if x < 0 else x


def my_sqrt(x):
    """Newton-Raphson square root."""
    if x < 0:
        raise ValueError("Square root of a negative number.")
    if x == 0:
        return 0.0

    guess = x if x >= 1 else 1.0

    for _ in range(30):
        new_guess = 0.5 * (guess + x / guess)
        if absolute(new_guess - guess) < 1e-12:
            break
        guess = new_guess

    return guess


def my_exp(x):
    """
    Manual exponential using:
        e^x = (e^(x/n))^n
    and a Taylor series for the small value x/n.
    """
    # Reduce the magnitude first so the Taylor series converges quickly.
    n = 32
    y = x / n

    # e^y using Taylor series.
    term = 1.0
    total = 1.0

    for k in range(1, 60):
        term = term * y / k
        total += term
        if absolute(term) < 1e-16:
            break

    # Raise e^(x/n) to the nth power.
    result = total
    for _ in range(n - 1):
        result *= total

    return result


def my_tanh(x):
    """tanh(x) = (e^(2x)-1)/(e^(2x)+1)."""
    if x > 20:
        return 1.0
    if x < -20:
        return -1.0

    e2x = my_exp(2.0 * x)
    return (e2x - 1.0) / (e2x + 1.0)


def gelu(x):
    """
    GELU approximation used in the PDF:
        GELU(x) = x/2 * [1 + tanh(sqrt(2/pi) * (x + 0.044715*x^3))]

    sqrt(2/pi) is written as a constant so no library is needed.
    """
    SQRT_2_OVER_PI = 0.7978845608028654
    return 0.5 * x * (
        1.0 + my_tanh(
            SQRT_2_OVER_PI * (x + 0.044715 * x * x * x)
        )
    )


def softmax_row(row):
    """
    Numerically stable manual softmax.
    Subtracting the largest value does not change the final probabilities.
    """
    largest = row[0]

    for x in row:
        if x > largest:
            largest = x

    exponentials = []
    for x in row:
        exponentials.append(my_exp(x - largest))

    total = 0.0
    for value in exponentials:
        total += value

    return [value / total for value in exponentials]


def softmax_matrix(A):
    return [softmax_row(row) for row in A]


def layer_norm_row(row, epsilon=1e-8):
    """LayerNorm for one vector."""
    d = len(row)

    # Mean
    mean = sum(row) / d

    # Variance
    variance = 0.0
    for x in row:
        variance += (x - mean) ** 2
    variance /= d

    # Standard deviation
    std = my_sqrt(variance + epsilon)

    # Normalize
    return [(x - mean) / std for x in row]


def layer_norm(A):
    return [layer_norm_row(row) for row in A]


def print_matrix(name, A, digits=4):
    print("\n" + name)
    for row in A:
        print("  [", end="")
        for j, value in enumerate(row):
            if j > 0:
                print(", ", end="")
            print(f"{value:.{digits}f}", end="")
        print("]")


def print_vector(name, v, digits=4):
    print(f"\n{name}")
    print("  [", end="")
    for i, value in enumerate(v):
        if i > 0:
            print(", ", end="")
        print(f"{value:.{digits}f}", end="")
    print("]")


# ------------------------------------------------------------
# Step 1-4: Raw text -> preprocessing -> tokens -> IDs
# ------------------------------------------------------------

print("=" * 70)
print("MANUAL TRANSFORMER ENCODER / SELF-ATTENTION")
print("=" * 70)

raw_text = "OMG!!! AI students rrr brillianttt 😊"
preprocessed_text = "ai students are brilliant"
tokens = ["ai", "students", "are", "brilliant"]
token_ids = [101, 205, 306, 412]

print("\nSTEP 1 - Raw Text")
print("T0 =", raw_text)

print("\nSTEP 2 - Preprocessing")
print("T1 =", preprocessed_text)

print("\nSTEP 3 - Tokenization")
print("Tokens =", tokens)

print("\nSTEP 4 - Token IDs")
print("IDs =", token_ids)


# ------------------------------------------------------------
# Step 5: Embedding lookup
# ------------------------------------------------------------

embedding_table = {
    101: [0.20, 0.40, 0.10, 0.50],  # ai
    205: [0.60, 0.10, 0.80, 0.30],  # students
    306: [0.10, 0.70, 0.20, 0.90],  # are
    412: [0.90, 0.30, 0.60, 0.20],  # brilliant
}

X = [embedding_table[token_id] for token_id in token_ids]

print_matrix("STEP 5 - Embedding Matrix X", X)


# ------------------------------------------------------------
# Step 6: Positional Encoding
# ------------------------------------------------------------
# d_model = 4
#
# PE(pos, 0) = sin(pos / 10000^(0/4)) = sin(pos)
# PE(pos, 1) = cos(pos / 10000^(0/4)) = cos(pos)
# PE(pos, 2) = sin(pos / 10000^(2/4)) = sin(pos/100)
# PE(pos, 3) = cos(pos / 10000^(2/4)) = cos(pos/100)
#
# Since this program cannot import math, the needed sin/cos values
# from the PDF are entered directly.

PE = [
    [0.0000,  1.0000, 0.0000, 1.0000],
    [0.8415,  0.5403, 0.0100, 0.99995],
    [0.9093, -0.4161, 0.0200, 0.9998],
    [0.1411, -0.9900, 0.0300, 0.9996],
]

Z = matrix_add(X, PE)

print_matrix("STEP 6 - Positional Encoding PE", PE, 5)
print_matrix("STEP 6.3 - Z = X + PE", Z, 5)


# ------------------------------------------------------------
# Step 7: Create Q, K, V
# ------------------------------------------------------------
#
# The matrices below are the projection matrices used in the PDF.
# d_model = 4
# d_k = d_v = 2

WQ = [
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [0.0, 1.0],
]

WK = [
    [1.0, 0.0],
    [0.0, 1.0],
    [0.5, 0.0],
    [0.0, 0.5],
]

WV = [
    [0.5, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [0.0, 0.5],
]

Q = matmul(Z, WQ)
K = matmul(Z, WK)
V = matmul(Z, WV)

print_matrix("STEP 7 - Query Matrix Q = ZWQ", Q, 5)
print_matrix("STEP 7 - Key Matrix K = ZWK", K, 5)
print_matrix("STEP 7 - Value Matrix V = ZWV", V, 5)


# ------------------------------------------------------------
# Step 8: Query-Key comparison
# ------------------------------------------------------------
# S = QK^T

KT = transpose(K)
S = matmul(Q, KT)

print_matrix("STEP 8 - Raw Score Matrix S = QK^T", S, 5)

print("\nInterpretation:")
print("S[i][j] tells how strongly token i's Query matches")
print("token j's Key. Larger score = stronger match.")


# ------------------------------------------------------------
# Step 9: Scale scores
# ------------------------------------------------------------
# d_k = 2
# S' = S / sqrt(d_k)

d_k = 2.0
sqrt_dk = my_sqrt(d_k)

S_scaled = matrix_scale(S, 1.0 / sqrt_dk)

print(f"\nSTEP 9 - Scaling")
print(f"sqrt(d_k) = sqrt(2) = {sqrt_dk:.6f}")
print_matrix("Scaled Score Matrix S' = S / sqrt(d_k)", S_scaled, 5)


# ------------------------------------------------------------
# Step 10: Softmax -> attention weights
# ------------------------------------------------------------

A = softmax_matrix(S_scaled)

print_matrix("STEP 10 - Attention Matrix A = softmax(S')", A, 6)

print("\nAttention percentages:")
for i, token in enumerate(tokens):
    print(f"\nQuery = {token}")
    for j, other_token in enumerate(tokens):
        print(f"  {other_token:10s}: {A[i][j] * 100:6.2f}%")

print("\nCheck first row:")
print(" + ".join(f"{x:.3f}" for x in A[0]), "=", f"{sum(A[0]):.3f}")


# ------------------------------------------------------------
# Step 11: Apply attention weights to V
# O = AV
# ------------------------------------------------------------

O = matmul(A, V)

print_matrix("STEP 11 - Attention Output O = AV", O, 6)


# ------------------------------------------------------------
# Step 12: Output projection
# H = OWO
# ------------------------------------------------------------

WO = [
    [1.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 1.0, 0.0],
]

H = matmul(O, WO)

print_matrix("STEP 12 - Output Projection H = OWO", H, 6)


# ------------------------------------------------------------
# Step 13: Residual connection
# R = Z + H
# ------------------------------------------------------------

R = matrix_add(Z, H)

print_matrix("STEP 13 - Residual R = Z + H", R, 6)


# ------------------------------------------------------------
# Step 14: Layer Normalization
# ------------------------------------------------------------

Y = layer_norm(R)

print_matrix("STEP 14 - LayerNorm(R)", Y, 6)

print("\nFor AI specifically:")
print_vector("Y_AI", Y[0], 6)


# ------------------------------------------------------------
# Step 15: Feed-Forward Network
# FFN(Y) = GELU(YW1)W2
#
# d_model = 4
# d_ff = 3
# ------------------------------------------------------------

W1 = [
    [1.0,  0.0, 1.0],
    [0.0,  1.0, 1.0],
    [1.0,  1.0, 0.0],
    [1.0, -1.0, 1.0],
]

W2 = [
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 1.0],
]

hidden = matmul(Y, W1)
activated = [[gelu(x) for x in row] for row in hidden]
FFN = matmul(activated, W2)

print_matrix("STEP 15.1 - Hidden = YW1", hidden, 6)
print_matrix("STEP 15.2 - GELU(Hidden)", activated, 6)
print_matrix("STEP 15.3 - FFN(Y) = GELU(YW1)W2", FFN, 6)


# ------------------------------------------------------------
# Step 16: Second residual connection
# R2 = Y + FFN(Y)
# ------------------------------------------------------------

R2 = matrix_add(Y, FFN)

print_matrix("STEP 16 - Second Residual R2 = Y + FFN(Y)", R2, 6)


# ------------------------------------------------------------
# Step 17: Second LayerNorm
# ------------------------------------------------------------

Z_out = layer_norm(R2)

print_matrix("STEP 17 - Final Output Zout = LayerNorm(R2)", Z_out, 6)


# ------------------------------------------------------------
# Final interpretation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL CONTEXT-AWARE REPRESENTATIONS")
print("=" * 70)

for i, token in enumerate(tokens):
    print(f"{token:10s} -> [", end="")
    for j, value in enumerate(Z_out[i]):
        if j > 0:
            print(", ", end="")
        print(f"{value:.4f}", end="")
    print("]")

print("\nThe final vector for each word is context-aware because")
print("the Self-Attention step allowed every token to collect")
print("information from the other tokens using Q -> K -> V.")
print("\nNote:")
print("The PDF rounds several intermediate values and also uses")
print("teaching approximations for GELU. Therefore, tiny numerical")
print("differences from the PDF's displayed values are expected.")

MANUAL TRANSFORMER ENCODER / SELF-ATTENTION

STEP 1 - Raw Text
T0 = OMG!!! AI students rrr brillianttt 😊

STEP 2 - Preprocessing
T1 = ai students are brilliant

STEP 3 - Tokenization
Tokens = ['ai', 'students', 'are', 'brilliant']

STEP 4 - Token IDs
IDs = [101, 205, 306, 412]

STEP 5 - Embedding Matrix X
  [0.2000, 0.4000, 0.1000, 0.5000]
  [0.6000, 0.1000, 0.8000, 0.3000]
  [0.1000, 0.7000, 0.2000, 0.9000]
  [0.9000, 0.3000, 0.6000, 0.2000]

STEP 6 - Positional Encoding PE
  [0.00000, 1.00000, 0.00000, 1.00000]
  [0.84150, 0.54030, 0.01000, 0.99995]
  [0.90930, -0.41610, 0.02000, 0.99980]
  [0.14110, -0.99000, 0.03000, 0.99960]

STEP 6.3 - Z = X + PE
  [0.20000, 1.40000, 0.10000, 1.50000]
  [1.44150, 0.64030, 0.81000, 1.29995]
  [1.00930, 0.28390, 0.22000, 1.89980]
  [1.04110, -0.69000, 0.63000, 1.19960]

STEP 7 - Query Matrix Q = ZWQ
  [0.30000, 2.90000]
  [2.25150, 1.94025]
  [1.22930, 2.18370]
  [1.67110, 0.50960]

STEP 7 - Key Matrix K = ZWK
  [0.25000, 2.15000]
  [1.84650, 1.290